# Ensembla Cast Enricher — Kaggle GPU Runner

Runs the PaddleOCR-first + Qwen3-VL-fallback pipeline on a free Kaggle **T4 GPU**.
Kaggle's IPs are not throttled by YouTube (the local-laptop blocker) and the GPU
runs the VLM at ~1-3s/frame instead of ~5 min on CPU.

### Before you run (do these once, in the panel on the right):
1. **Settings → Accelerator → GPU T4 x1**
2. **Settings → Internet → On**
3. **Add-ons → Secrets**, add these (Label = value):
   - `VITE_SUPABASE_URL`
   - `SUPABASE_SERVICE_ROLE_KEY`
   - `GEMINI_API_KEY`
   - `GITHUB_TOKEN`  — a GitHub PAT with read access to the private `muvidb` repo
   - *(optional)* `GROQ_API_KEY`, `OPENAI_API_KEY`

Then **Run All**. Cell 6 validates one film; uncomment the last cell for the 24/7 loop.

In [ ]:
# 1. System deps: zstd (Ollama installer needs it), Node (yt-dlp JS), ffmpeg (preinstalled on Kaggle)
import subprocess, time, os, urllib.request
subprocess.run('apt-get -qq update && apt-get -qq install -y nodejs zstd', shell=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
print('ffmpeg:', subprocess.run('ffmpeg -version', shell=True, capture_output=True, text=True).stdout.splitlines()[0])
print('node:', subprocess.run('node --version', shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
# 2. Start Ollama and pull the vision model (~3 GB; takes a few minutes once per session)
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['ollama', 'serve'])
for _ in range(30):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434', timeout=2); break
    except Exception:
        time.sleep(2)
subprocess.run(['ollama', 'pull', 'qwen3-vl:4b-instruct'], check=True)
print('Ollama + model ready')

In [ ]:
# 3. Python deps
!pip -q install paddleocr paddlepaddle yt-dlp google-genai requests python-dotenv rapidfuzz imagehash opencv-python-headless

In [ ]:
# 4. Secrets -> environment
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()

def load(key, required=True):
    try:
        os.environ[key] = sec.get_secret(key)
    except Exception:
        if required:
            print(f'  MISSING required secret: {key}')
        return False
    return True

load('VITE_SUPABASE_URL')
load('SUPABASE_SERVICE_ROLE_KEY')
load('GEMINI_API_KEY', required=False)
load('GROQ_API_KEY', required=False)
load('OPENAI_API_KEY', required=False)

# Enable the VLM rescue (we have a GPU here) and pin the model tag.
os.environ['ENABLE_VLM_FALLBACK'] = '1'
os.environ['OLLAMA_VISION_MODEL'] = 'qwen3-vl:4b-instruct'
os.environ['OLLAMA_MODEL'] = 'qwen3-vl:4b-instruct'
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
print('env configured')

In [ ]:
# 5. Clone the private repo (staging branch) using the GITHUB_TOKEN secret
token = ''
try:
    token = sec.get_secret('GITHUB_TOKEN') + '@'
except Exception:
    print('No GITHUB_TOKEN secret found - clone will fail unless the repo is public.')
url = f'https://{token}github.com/Amichael10/muvidb.git'
subprocess.run(['rm', '-rf', 'muvidb'])
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'staging', url], check=True)
print('repo cloned')

In [ ]:
# 6. VALIDATE on one film (full hybrid: PaddleOCR + GPU VLM rescue).
#    Watch the output for clean cast/crew names and the saved markdown.
subprocess.run(
    ['python', 'local_ocr_extractor.py', 'https://www.youtube.com/watch?v=fGlUUt1xdkw'],
    cwd='muvidb'
)

In [ ]:
# 7. PRODUCTION: 24/7 backward sweep over the whole DB (page 1 -> oldest).
#    Uncomment to run. Note: a Kaggle session lasts up to ~9-12h, so re-trigger
#    (or use 'Save & Run All' commits) for true 24/7.
# subprocess.run(['python', 'local_ocr_extractor.py', '--loop'], cwd='muvidb')